# Create Open Society Foundations Awards

Creates Open Society Foundations grant records from
opensocietyfoundations.org/grants/past, scraped via Playwright +
headless Chromium (the page is JS-rendered with no public API).
OSF is a major international philanthropic funder ($150M+/yr).

**Prerequisites:**
- Run `scripts/local/osf_to_s3.py` (Playwright) to scrape the listing
  pages and upload the parquet first.

**Data source:** opensocietyfoundations.org/grants/past?page=N
**S3 location:** `s3a://openalex-ingest/awards/osf/osf_grants.parquet`

**OSF funder (OpenAlex):**
- funder_id: 4320306189
- ROR: https://ror.org/00qnfvz68
- DOI: 10.13039/100000919
- display_name: "Open Society Foundations"

**Schema notes:**
- Each listing card exposes only grantee + year + amount. There is
  no project description or PI on the public listing — those would
  require navigating to per-grant detail pages (potentially future
  enrichment).
- Currency hardcoded USD. Year is stored as start_year (single value
  per grant; OSF's listing doesn't separate start/end).
- `lead_investigator.affiliation.name` = the grantee org. No PI name.


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.osf_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/osf/osf_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.osf_raw;

## Step 1.5: Inspect raw + verify amount/currency

In [ ]:
%sql
DESCRIBE openalex.awards.osf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.osf_raw LIMIT 5;

In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(amount_usd) AS has_amount,
  ROUND(MIN(amount_usd), 0) AS min_amt,
  ROUND(MAX(amount_usd), 0) AS max_amt,
  ROUND(AVG(amount_usd), 0) AS avg_amt
FROM openalex.awards.osf_raw;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.osf_awards
USING delta
AS
WITH osf_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306189
),
src AS (
    -- Multiple grants to the same grantee in the same year are common; build
    -- a stable surrogate key from grantee + year + amount + page index.
    SELECT
        *,
        abs(xxhash64(CONCAT(
            COALESCE(grantee_name, ''), ':', CAST(year AS STRING), ':',
            COALESCE(CAST(amount_usd AS STRING), ''), ':', CAST(page AS STRING),
            ':', CAST(monotonically_increasing_id() AS STRING)
        ))) % 9000000000 AS surrogate_id
    FROM openalex.awards.osf_raw
    WHERE grantee_name IS NOT NULL AND TRIM(grantee_name) != ''
)
SELECT
    -- Hash funder_id with the surrogate so the OpenAlex ID is stable across re-runs
    abs(xxhash64(CONCAT(f.funder_id, ':osf:', CAST(s.surrogate_id AS STRING)))) % 9000000000 AS id,
    -- "{Grantee} ({YYYY})" gives a readable display
    CONCAT(s.grantee_name, ' (', CAST(s.year AS STRING), ')') AS display_name,
    CAST(NULL AS STRING) AS description,
    f.funder_id,
    CAST(s.surrogate_id AS STRING) AS funder_award_id,
    s.amount_usd AS amount,
    'USD' AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    CAST(NULL AS STRING) AS funder_scheme,
    'osf_grants_past' AS provenance,
    -- Listing exposes only year; default start = Jan 1, end = Dec 31 of that year
    TRY_TO_DATE(CONCAT(s.year, '-01-01'), 'yyyy-MM-dd') AS start_date,
    TRY_TO_DATE(CONCAT(s.year, '-12-31'), 'yyyy-MM-dd') AS end_date,
    s.year AS start_year,
    s.year AS end_year,
    -- OSF funds organisations; no PI on public listing
    struct(
        CAST(NULL AS STRING) AS given_name,
        CAST(NULL AS STRING) AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            s.grantee_name AS name,
            CAST(NULL AS STRING) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    CONCAT('https://www.opensocietyfoundations.org/grants/past?page=', CAST(s.page AS STRING)) AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':osf:', CAST(s.surrogate_id AS STRING)))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM src s
CROSS JOIN osf_funder f;

## Step 3: Insert into openalex_awards_raw at priority 45

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'osf_grants_past' AND priority = 45;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    45 AS priority
FROM openalex.awards.osf_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.osf_awards;

In [ ]:
%sql
-- Step 6.7 fail-fast amount/currency check
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(amount), 0) AS avg_amt
FROM openalex.awards.osf_awards;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 70) AS title,
       amount, start_year,
       lead_investigator.affiliation.name AS grantee
FROM openalex.awards.osf_awards
ORDER BY amount DESC LIMIT 10;

In [ ]:
%sql
SELECT start_year, COUNT(*) AS cnt, ROUND(SUM(amount), 0) AS total_usd
FROM openalex.awards.osf_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC;